In [1]:
import pandas as pd

In [2]:
import json

with open('ignore/Community_Districts.geojson') as file:
    geojson_data = json.load(file)
len(geojson_data['features'])

71

In [3]:
dataset=pd.read_csv('ignore/languages.csv')
dataset[dataset.columns[0]].value_counts()

American Community Survey (ACS) Data Time Period
2015-2019    8024
Name: count, dtype: int64

In [4]:
columns=dataset.columns
drop_columns=[c for i,c in enumerate(columns) if i in [0,6,8]]
dataset.drop(columns=drop_columns,inplace=True)
dataset.columns

Index(['Borough', 'Borough Community District Code', 'Community District Name',
       'Language', 'LEP Population (Estimate)',
       'CVALEP Population (Estimate)'],
      dtype='object')

In [5]:
boroughs=dataset[['Borough','Borough Community District Code']].copy()
boroughs['borough_id']=boroughs['Borough Community District Code'].apply(lambda x:str(x)[0:1])
boroughs=boroughs[['Borough','borough_id']].drop_duplicates().set_index('borough_id')
boroughs.index.name=''
boroughs=boroughs['Borough']
boroughs


1        Manhattan
2            Bronx
3         Brooklyn
4           Queens
5    Staten Island
Name: Borough, dtype: object

In [6]:
columns=dataset.columns
select_columns=[c for i,c in enumerate(columns) if i in [0,1,2]]
community_districts=dataset[select_columns].copy().drop_duplicates().set_index(columns[1])
community_districts.index.name=''
community_districts.sort_index(inplace=True)
community_districts['Borough']=community_districts['Borough'].apply(lambda x:boroughs[boroughs==x].index[0])
community_districts

,Borough,Community District Name
,,
101,1,"Battery Park City, Tribeca"
102,1,"Greenwich Village, Soho"
103,1,"Lower East Side, Chinatown"
104,1,"Chelsea, Clinton"
105,1,Midtown Business District
106,1,"Stuyvesant Town, Turtle Bay"
107,1,"West Side, Upper West Side"
108,1,Upper East Side
109,1,"Manhattanville, Hamilton Heights"


In [7]:
print(dataset.count())
dataset=dataset[dataset['LEP Population (Estimate)']>0].copy()
dataset.count()

Borough                            8024
Borough Community District Code    8024
Community District Name            8024
Language                           8024
LEP Population (Estimate)          8024
CVALEP Population (Estimate)       8024
dtype: int64


Borough                            2290
Borough Community District Code    2290
Community District Name            2290
Language                           2290
LEP Population (Estimate)          2290
CVALEP Population (Estimate)       2290
dtype: int64

In [8]:
languages=dataset['Language'].copy().drop_duplicates()
languages.reset_index(drop=True,inplace=True)
languages

0                                 Albanian
1                                Bulgarian
2                                  Burmese
3      Chinese (incl. Mandarin, Cantonese)
4                                 Filipino
                      ...                 
106             Other Philippine languages
107                                Kannada
108                    Muskogean languages
109                                 Samoan
110                  Uto-Aztecan languages
Name: Language, Length: 111, dtype: object

In [9]:
dataset.columns

Index(['Borough', 'Borough Community District Code', 'Community District Name',
       'Language', 'LEP Population (Estimate)',
       'CVALEP Population (Estimate)'],
      dtype='object')

In [10]:
populations=dataset[['Borough Community District Code','Language','LEP Population (Estimate)','CVALEP Population (Estimate)']].copy()
populations['Language']=populations['Language'].apply(lambda x:languages[languages==x].index[0])
populations.rename(columns={'Language':'language_id',
                          'Borough Community District Code':'borough_id',
                          'LEP Population (Estimate)':'lep_population',
                          'CVALEP Population (Estimate)':'cvalep_population'},inplace=True)
populations.reset_index(drop=True,inplace=True)
populations

,borough_id,language_id,lep_population,cvalep_population
0,101,0,7,0
1,101,1,68,31
2,101,2,19,19
3,101,3,1946,1150
4,101,4,14,0
...,...,...,...,...
2285,503,25,50,50
2286,503,51,36,0
2287,503,45,99,53
2288,503,28,172,129


In [11]:
print(boroughs)
print(community_districts)
print(languages)
print(populations)


1        Manhattan
2            Bronx
3         Brooklyn
4           Queens
5    Staten Island
Name: Borough, dtype: object
    Borough              Community District Name
                                                
101       1           Battery Park City, Tribeca
102       1              Greenwich Village, Soho
103       1           Lower East Side, Chinatown
104       1                     Chelsea, Clinton
105       1            Midtown Business District
106       1          Stuyvesant Town, Turtle Bay
107       1           West Side, Upper West Side
108       1                      Upper East Side
109       1     Manhattanville, Hamilton Heights
110       1                       Central Harlem
111       1                          East Harlem
112       1           Washington Heights, Inwood
201       2     Melrose, Mott Haven, Port Morris
202       2                Hunts Point, Longwood
203       2        Morrisania, Crotona Park East
204       2        Highbridge, Concourse V

In [12]:
from mysite import settings
import django
import os
os.environ['DJANGO_SETTINGS_MODULE'] = 'mysite.settings'
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()
from esl_ny.models import Borough,CommunityDistrict,Language,LEPPopulation




In [13]:
Borough.objects.all().delete()
for id,name in boroughs.to_dict().items():
    Borough.objects.create(id=id,name=name)
Borough.objects.all()

<QuerySet [<Borough: Manhattan>, <Borough: Bronx>, <Borough: Brooklyn>, <Borough: Queens>, <Borough: Staten Island>]>

In [15]:
CommunityDistrict.objects.all().delete()
for id,dict in community_districts.to_dict(orient='index',index=True).items():
    name=dict['Community District Name']
    borough=Borough.objects.get(pk=dict['Borough'])
    new_cd=CommunityDistrict(id=id,borough=borough,name=name)
    new_cd.geojson=list(filter(lambda x:x['properties']['boro_cd']==str(id),geojson_data['features']))[0]
    new_cd.save()

In [16]:
Language.objects.all().delete()
for id,name in languages.to_dict().items():
    Language.objects.create(id=id,name=name)
    print(id,name)

0 Albanian
1 Bulgarian
2 Burmese
3 Chinese (incl. Mandarin, Cantonese)
4 Filipino
5 French
6 German
7 Greek
8 Gujarati
9 Hebrew
10 Hindi
11 Hungarian
12 India N.E.C.
13 Italian
14 Japanese
15 Khmer
16 Korean
17 Macedonian
18 Min Nan Chinese
19 Other languages of Africa
20 Polish
21 Portuguese
22 Romanian
23 Russian
24 Spanish
25 Tagalog
26 Telugu
27 Thai
28 Ukrainian
29 Urdu
30 Vietnamese
31 Arabic
32 Bengali
33 Haitian
34 Malay
35 Malayalam
36 Other Eastern Malayo-Polynesian languages
37 Pakistan N.E.C.
38 Serbocroatian
39 Swedish
40 Indonesian
41 Nepali
42 Norwegian
43 Other Afro-Asiatic languages
44 Somali
45 Turkish
46 Yiddish
47 Croatian
48 Farsi
49 Serbian
50 Slovak
51 Tamil
52 Tigrinya
53 Yoruba
54 Akan (incl. Twi)
55 Amharic
56 Bosnian
57 Igbo
58 Punjabi
59 Czech
60 Finnish
61 Fulah
62 Other Central and South American languages
63 Other Bantu languages
64 Other Indo-European languages
65 Other Mande languages
66 Other Niger-Congo languages
67 Other and unspecified languages
68 

In [17]:
LEPPopulation.objects.all().delete()
for each in populations.to_dict(orient='records'):
    district=CommunityDistrict.objects.get(pk=each['borough_id'])
    language=Language.objects.get(pk=each['language_id'])
    # print(district_id,language_id,each)
    LEPPopulation.objects.create(communitydistrict=district,
                                 language=language,
                                lep_population=each['lep_population'],
                                cvalep_population=each['cvalep_population'])